In [ ]:
def _initiate_actors(self, pg, num_gpus_per_actor):
        world_size = self._num_nodes * self._num_gpus_per_node

        # Use placement group to lock resources for models of same type
        if self._num_gpus_per_node > 1 and pg is None:
            bundles = [
                {"GPU": self._num_gpus_per_node, "CPU": self._num_gpus_per_node} for _ in range(self._num_nodes)
            ]
            if self._resources:
                resources_name = list(self._resources.keys())[0]
                for i in range(len(bundles)):
                    bundles[i][resources_name] = self._num_resources_per_node

            pg = placement_group(bundles, strategy="STRICT_SPREAD")
            ray.get(pg.ready())
        if pg:
            master_actor = self.ray_actor_type.options(
                num_cpus=num_gpus_per_actor,
                num_gpus=num_gpus_per_actor,
                resources=self._resources,
                scheduling_strategy=PlacementGroupSchedulingStrategy(
                    placement_group=pg, placement_group_bundle_index=0
                ),
            ).remote(world_size, 0, None, None)
        else:
            master_actor = self.ray_actor_type.options(
                num_cpus=num_gpus_per_actor,
                num_gpus=num_gpus_per_actor,
                resources=self._resources,
            ).remote(world_size, 0, None, None)
        self._actor_handlers = [master_actor]

        # Create worker actors
        if world_size > 1:
            master_addr, master_port = ray.get(master_actor.get_master_addr_port.remote())
            for rank in range(1, world_size):
                if pg:
                    worker_actor = self.ray_actor_type.options(
                        num_cpus=num_gpus_per_actor,
                        num_gpus=num_gpus_per_actor,
                        resources=self._resources,
                        scheduling_strategy=PlacementGroupSchedulingStrategy(
                            placement_group=pg,
                            placement_group_bundle_index=rank // self._num_gpus_per_node,
                        ),
                    ).remote(world_size, rank, master_addr, master_port)
                else:
                    worker_actor = self.ray_actor_type.options(
                        num_cpus=num_gpus_per_actor,
                        num_gpus=num_gpus_per_actor,
                        resources=self._resources,
                    ).remote(world_size, rank, master_addr, master_port)
                self._actor_handlers.append(worker_actor)

# Ray 分布式训练启动器（Launcher） 的核心逻辑 

这段代码是一个典型的 Ray 分布式训练启动器（Launcher） 的核心逻辑。
- <font color='red'>它的主要职责是在 Ray 集群中初始化并管理一组分布式 Actor（通常代表训练进程），并确保它们按照预期的拓扑结构（如多机多卡）进行资源绑定和通信。</font>
- <font color='green'>这段代码展示了如何利用 Ray 的 Placement Group (资源组) 来实现严格的资源隔离和分布式训练环境的搭建。</font>

以下是这段代码的详细逐层解析：

##  核心目标：构建分布式训练集群

这段代码的目的是创建一个由 world_size 个 Actor 组成的集群。这些 Actor 通常对应于分布式训练中的不同 Rank（进程），它们需要协同工作（例如使用 DeepSpeed 或 Megatron-LM 进行训练）。
##  关键逻辑拆解
📦 资源组的创建与策略 (Placement Group)

In [ ]:
if self._num_gpus_per_node > 1 and pg is None:
    bundles = [
        {"GPU": self._num_gpus_per_node, "CPU": self._num_gpus_per_node} for _ in range(self._num_nodes)
    ]
    #  资源组的创建
    pg = placement_group(bundles, strategy="STRICT_SPREAD")
    ray.get(pg.ready())

- 目的：<font color='red'>这是 Ray 分布式调度的核心技巧。</font>
- Bundles (资源束)：代码构建了一个资源列表。每个 Bundle 代表一个物理节点所需的资源（例如一台机器有 8 张 GPU，就申请 8 个 GPU 资源）。
- STRICT_SPREAD 策略：这是一个非常关键的参数。<font color='red'>它强制 Ray 将不同的 Bundle 分散 到不同的物理节点上。</font>
  - 如果你需要 2 台机器，每台 8 卡，Ray 会确保这 16 张卡分布在 2 台不同的机器上，而不是挤在同一台机器上。这对于多机训练至关重要。
  
### 🌟 创建 Master Actor (Rank 0) 它通常负责协调其他节点、保存检查点或作为通信的根节点。

In [ ]:
if pg:
    master_actor = self.ray_actor_type.options(
        # PlacementGroupSchedulingStrategy
        scheduling_strategy=PlacementGroupSchedulingStrategy(
            placement_group=pg,
            placement_group_bundle_index=0 # 强制将这个 Master Actor 调度到资源组的 第一个 Bundle（即第一台机器的第一组资源）中。
        ),
    ).remote(world_size, 0, None, None)

- <b>角色</b>：这是分布式训练的主节点（Rank 0）。它通常负责协调其他节点、保存检查点或作为通信的根节点。
- <b>调度</b>：通过 placement_group_bundle_index=0，强制将这个 Master Actor 调度到资源组的 第一个 Bundle（即第一台机器的第一组资源）中。
- <b>初始化参数</b>： <font color='red'>.remote(world_size, 0, None, None) </font><font color='green'>告诉 Actor：“你是第 0 号进程，总共有 world_size 个进程，目前没有 Master 地址（因为你自己就是）”。</font>

### 👷 创建 Worker Actors (Rank 1 to N) 

In [ ]:
master_addr, master_port = ray.get(master_actor.get_master_addr_port.remote())

for rank in range(1, world_size):
    # ...
    worker_actor = self.ray_actor_type.options(
        # PlacementGroupSchedulingStrategy
        scheduling_strategy=PlacementGroupSchedulingStrategy(
            placement_group=pg,
            placement_group_bundle_index=rank // self._num_gpus_per_node,
        ),
    ).remote(world_size, rank, master_addr, master_port) 

- <b>获取 Master 信息</b>：Worker 需要知道 Master 在哪里才能加入训练集群（建立分布式通信组，如 Torch Distributed），所以先从 Master Actor 获取 IP 和端口。
- <b>智能调度 (rank // num_gpus_per_node)：</b>
    - <font color='red'>这是一个计算 Bundle 索引的数学技巧。</font>
    - 假设每台机器有 8 张卡 (num_gpus_per_node = 8)。
    - Rank 1 到 Rank 7：<font color='red'>1//8 到 7//8 结果都是 0</font>。这意味着它们会被调度到 第 0 号 Bundle（第一台机器），与 Master 在一起。
    - Rank 8 到 Rank 15：<font color='red'>8//8 到 15//8 结果都是 1</font>。这意味着它们会被调度到 第 1 号 Bundle（第二台机器）。
    - 结果：这完美地将分布式进程映射到了物理拓扑上，确保了同一台机器上的进程优先通信（通常通过 NVLink），跨机器进程通过网络通信。
    
🎯 资源请求 (num_gpus_per_actor)

In [ ]:
num_cpus=num_gpus_per_actor,
num_gpus=num_gpus_per_actor,

- 这里向 Ray 声明每个 Actor 需要多少 GPU。
- 逻辑资源与物理资源：正如之前的对话提到的，Ray 管理的是逻辑资源。这里请求的 num_gpus_per_actor 可能是一个分数（如 0.5 或 0.25），用于实现单卡多模型（Co-located）的部署模式，或者是一个整数（如 1.0）用于独占模式。

## 总结：这段代码在做什么？

想象你是一个包工头（Driver 进程），这段代码就是你的施工计划：

1. 圈地：首先，你向 Ray 申请了 num_nodes 块地皮（Placement Group），并要求这些地皮必须分散在不同的区域（STRICT_SPREAD），不能挤在一起。
2. 招工头：你在第一块地皮的第一个位置设立了一个“工头”（Master Actor），让他负责喊话。
3. 招工人：然后你根据工头提供的联系方式，招募剩下的工人（Worker Actors）。
4. 分配宿舍：你通过计算（rank // gpus_per_node），精确地把工人分配到对应的地皮（Bundle）上，确保每张床（GPU）上都有人，且同一宿舍（物理节点）的人能互相照应。

这种模式在 Slime、OpenRLHF 等大规模强化学习框架中被广泛使用，用于构建高性能的训推一体化集群。

# placement_group 放置组
- from ray.util.placement_group import PlacementGroup, placement_group

###### ray.util.placement_group 中的 placement_group 函数是 Ray 框架中用于实现原子性资源预留的核心工具。
- <font color ='red'>它允许你将一组资源（如 CPU、GPU）打包成一个或多个“资源束”（Bundle），并以原子方式在集群中预留这些资源，确保相关的任务（Task）或参与者（Actor）能够被调度到符合预期的物理节点上。</font>

简单来说，Placement Group 是解决分布式计算中“组调度”（Gang Scheduling）问题的关键，即一组任务要么全部成功分配资源并启动，要么全部不启动，避免了因部分资源不足导致的任务死锁或性能下降。

## 🧩 核心概念：资源束 (Bundle)

在深入 placement_group 函数之前，必须先理解 Bundle。

- 定义：<font color ='red'>Bundle 是一个资源需求的字典，例如 {"CPU": 2, "GPU": 1}。</font>
- 特性：一个 Bundle 中的所有资源必须被调度到同一个物理节点上。你不能创建一个 {"GPU": 10} 的 Bundle 并期望它分布在多个只有 8 张 GPU 的节点上。
- 作用：它是资源预留的最小单元。<font color ='red'>Placement Group 本质上就是一组 Bundle 的集合。</font>

## ⚙️ placement_group 函数详解

该函数用于创建一个 Placement Group。它的主要参数和返回值如下：

### 主要参数
- <font color ='red'>bundles (列表):</font>

    - 这是最核心的参数，一个包含多个资源需求字典（Bundle）的列表。
    - 示例：[{"GPU": 8}<font color ='red'>, </font>{"GPU": 8}] <font color ='red'>表示申请两个 Bundle，每个 Bundle 包含 8 个 GPU。</font>这通常用于多机训练，确保两个节点各有 8 张卡可用。
    
- <font color ='red'>strategy (字符串):</font>

<font color ='red'>定义了多个 Bundle 之间的调度策略，决定了它们是倾向于聚集在一起还是分散开。</font>

   - <font color ='red'>PACK (默认):</font> <font color ='green'>尽可能将所有 Bundle 调度到同一个节点上。</font>这有利于数据局部性，减少跨节点通信开销。
   - <font color ='red'>STRICT_PACK:</font> 强制所有 Bundle 必须在同一个节点上。如果无法满足，整个 Placement Group 的创建将失败。
   - <font color ='red'>SPREAD:</font> <font color ='green'>尽可能将每个 Bundle 调度到不同的节点上。</font>这有利于负载均衡和容错。
   - <font color ='red'>STRICT_SPREAD:</font> 强制每个 Bundle 必须在不同的节点上。这是多机分布式训练的典型需求，确保每个节点只运行一个 Bundle 的任务。
###  返回值
- 函数返回一个 PlacementGroup 对象。这个对象本身只是一个“占位符”或“预约单”，<font color ='red'>代表了资源的预留请求。</font>
- 创建过程是异步的。<font color ='red'>你需要调用 ray.get(pg.ready()) 来阻塞等待，直到 Ray 集群成功预留了所有资源，或者因资源不足而失败。</font>

## 🔑 生命周期管理
- <b>自动回收</b>：Placement Group 的生命周期通常与其创建者（Job 或 Detached Actor）绑定。当创建它的 Driver 进程结束时，Placement Group 会自动被删除，预留的资源也会被释放。
- <b>手动删除</b>：你也可以通过 ray.util.placement_group.remove_placement_group(pg) 来手动删除一个 Placement Group。
- <b>资源释放</b>：一旦 Placement Group 被销毁，所有在其上运行的 Actor 和 Task 都会被终止。

##  使用 Placement Group 通常分为三步：创建、等待、调度。

In [ ]:
import ray
from ray.util.placement_group import placement_group
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy

# 1. 创建 Placement Group
# 假设我们需要在两个不同的节点上，每个节点运行一个需要 2 个 GPU 的任务
pg = placement_group(
    bundles=[
        {"GPU": 2, "CPU": 4},  # Bundle 0: 第一个节点的2个GPU
        {"GPU": 2, "CPU": 4},  # Bundle 1: 第二个节点的2个GPU
    ],
    strategy="STRICT_SPREAD"  # 强制分散到不同节点
)

# 2. 等待资源预留成功
# 这一步至关重要，确保资源真正到位后再提交任务
try:
    ray.get(pg.ready())
    print("Placement Group 创建成功！")
except Exception:
    print("资源不足，Placement Group 创建失败！")

# 3. 将 Actor 或 Task 调度到指定的 Bundle 上
@ray.remote(num_gpus=2)
class MyDistributedActor:
    def __init__(self, rank):
        self.rank = rank
    def get_rank(self):
        return self.rank

# 创建第一个 Actor，绑定到 Bundle 0 (placement_group_bundle_index=0)
actor_0 = MyDistributedActor.options(
    scheduling_strategy=PlacementGroupSchedulingStrategy(
        placement_group=pg,
        placement_group_bundle_index=0
    )
).remote(rank=0)

# 创建第二个 Actor，绑定到 Bundle 1 (placement_group_bundle_index=1)
actor_1 = MyDistributedActor.options(
    scheduling_strategy=PlacementGroupSchedulingStrategy(
        placement_group=pg,
        placement_group_bundle_index=1
    )
).remote(rank=1)

# 验证 Actor 是否成功创建
print(ray.get([actor_0.get_rank.remote(), actor_1.get_rank.remote()]))
# 输出: [0, 1]

In [ ]:
@PublicAPI
@client_mode_wrap
def placement_group(
    bundles: List[Dict[str, float]],
    strategy: str = "PACK",
    name: str = "",
    lifetime: Optional[str] = None,
    _soft_target_node_id: Optional[str] = None,
    bundle_label_selector: List[Dict[str, str]] = None,
) -> PlacementGroup:
    """Asynchronously creates a PlacementGroup.

    Args:
        bundles: A list of bundles which
            represent the resources requirements.
        strategy: The strategy to create the placement group.

         - "PACK": Packs Bundles into as few nodes as possible.
         - "SPREAD": Places Bundles across distinct nodes as even as possible.
         - "STRICT_PACK": Packs Bundles into one node. The group is
           not allowed to span multiple nodes.
         - "STRICT_SPREAD": Packs Bundles across distinct nodes.

        name: The name of the placement group.
        lifetime: Either `None`, which defaults to the placement group
            will fate share with its creator and will be deleted once its
            creator is dead, or "detached", which means the placement group
            will live as a global object independent of the creator.
        _soft_target_node_id: (Private, Experimental) Soft hint where bundles of
            this placement group should be placed.
            The target node is specified by it's hex ID.
            If the target node has no available resources or died,
            bundles can be placed elsewhere.
            This currently only works with STRICT_PACK pg.
        bundle_label_selector: A list of label selectors to apply to a
            placement group on a per-bundle level.

    Raises:
        ValueError: if bundle type is not a list.
        ValueError: if empty bundle or empty resource bundles are given.
        ValueError: if the wrong lifetime arguments are given.

    Return:
        PlacementGroup: Placement group object.
    """
    worker = ray._private.worker.global_worker
    worker.check_connected()

    validate_placement_group(
        bundles=bundles,
        strategy=strategy,
        lifetime=lifetime,
        _soft_target_node_id=_soft_target_node_id,
        bundle_label_selector=bundle_label_selector,
    )

    if bundle_label_selector is None:
        bundle_label_selector = []

    if lifetime == "detached":
        detached = True
    else:
        detached = False

    placement_group_id = worker.core_worker.create_placement_group(
        name,
        bundles,
        strategy,
        detached,
        _soft_target_node_id,
        bundle_label_selector,
    )

    return PlacementGroup(placement_group_id)

In [ ]:
@PublicAPI
class PlacementGroup:
    """A handle to a placement group."""

    @staticmethod
    def empty() -> "PlacementGroup":
        return PlacementGroup(PlacementGroupID.nil())

    def __init__(
        self,
        id: "ray._raylet.PlacementGroupID",
        bundle_cache: Optional[List[Dict]] = None,
    ):
        self.id = id
        self.bundle_cache = bundle_cache

    @property
    def is_empty(self):
        return self.id.is_nil()

    def ready(self) -> "ray._raylet.ObjectRef":
        """Returns an ObjectRef to check ready status.

        This API returns an ObjectRef that becomes ready when the placement group
        is created. It is compatible with ray.get, ray.wait, and await.

        Example:
            .. testcode::

                import ray

                pg = ray.util.placement_group([{"CPU": 1}])
                ray.get(pg.ready())

                pg = ray.util.placement_group([{"CPU": 1}])
                ray.wait([pg.ready()])

        """
        if self.is_empty:
            return ray.put(self)

        return _call_placement_group_ready_async(self)

    def wait(self, timeout_seconds: Union[float, int] = 30) -> bool:
        """Wait for the placement group to be ready within the specified time.
        Args:
             timeout_seconds(float|int): Timeout in seconds.
        Return:
             True if the placement group is created. False otherwise.
        """
        return _call_placement_group_ready(self.id, timeout_seconds)

    @property
    def bundle_specs(self) -> List[Dict]:
        """List[Dict]: Return bundles belonging to this placement group."""
        self._fill_bundle_cache_if_needed()
        return self.bundle_cache

    @property
    def bundle_count(self) -> int:
        self._fill_bundle_cache_if_needed()
        return len(self.bundle_cache)

    def _fill_bundle_cache_if_needed(self) -> None:
        if not self.bundle_cache:
            self.bundle_cache = _get_bundle_cache(self.id)

    def __eq__(self, other):
        if not isinstance(other, PlacementGroup):
            return False
        return self.id == other.id

    def __hash__(self):
        return hash(self.id)


# options() 是 Ray 框架中用于动态配置分布式任务（Task）和参与者（Actor）的核心功能

- @ray.remote 的 options 机制是 Ray 框架中用于动态配置分布式任务（Task）和参与者（Actor）的核心功能。
- 它允许你在不修改原始代码定义的情况下，灵活地调整资源需求、返回值数量或调度策略。
- 你可以把它理解为一种“运行时配置覆盖”机制：<font color='red'>它基于原始的远程函数或类，创建一个新的配置版本，而不会影响原始定义。</font>

## 🎯 核心作用：为什么需要 options？
- <font color='red'>通常我们在定义远程函数时会写死一些配置</font>，例如：

In [ ]:
@ray.remote(num_gpus=1)
def train():
    pass

###### 但在实际运行中，你可能希望：
1. <b>临时加资源：</b> 某次特定的调用需要 2 张 GPU 而不是默认的 1 张。
2. <b>动态改返回值：</b> 根据业务逻辑，有时需要返回 1 个结果，有时需要返回多个。
3. <b>指定调度位置</b> ：在多机训练中，强制将某个任务调度到特定的资源组（Placement Group）中。

<font color='red'>options 就是为了解决这些动态变化的需求而生的。</font>

## ⚙️ 两种调用方式

Ray 提供了两种使用 options 的方式，分别适用于不同的场景：

### 1. 装饰器参数 (静态定义)
直接在 @ray.remote 装饰器中定义参数。这是最常见的方式，适用于配置相对固定的场景。

In [ ]:
# 定义时就确定需要 2 个 CPU
@ray.remote(num_cpus=2)
def func():
    return 1

### 2. .options() 方法 (动态覆盖)
- <font color='red'>在调用 .remote() 之前，先调用 .options() 方法。</font>
- <font color='green'>这会返回一个新的可调用对象，该对象拥有更新后的配置。原始函数的配置不会被改变。</font>

In [ ]:
# func 默认可能需要 1 个 CPU
@ray.remote(num_cpus=1)
def func():
    return 1

# 使用 .options() 临时申请 4 个 CPU
# 这不会影响 func 的原始定义,只是临时修改!
heavy_task = func.options(num_cpus=4)
ray.get(heavy_task.remote()) 

## 📋 常用参数详解
options 支持绝大多数 @ray.remote 的参数，以下是高频使用的参数

| 参数类别 | 参数名 | 说明 | 示例 |
| :--- | :--- | :--- | :--- |
| 资源需求 | `num_cpus` | 指定任务所需的 CPU 核数。 | `.options(num_cpus=4)` |
| | `num_gpus` | 指定任务所需的 GPU 数量。 | `.options(num_gpus=0.5)` |
| | `memory` | 指定所需的内存大小（字节）。 | `.options(memory=1024*1024*100)` |
| | `resources` | 指定自定义资源（如 TPU, NPU）。 | `.options(resources={"TPU": 1})` |
| 返回值 | `num_returns` | 指定函数返回的对象引用（ObjectRef）数量。可以是整数或 `"dynamic"`。 | `.options(num_returns=2)` |
| 调度策略 | `scheduling_strategy` | <font color='red'>高级调度策略，常用于配合 Placement Group。</font> | `.options(scheduling_strategy=...)` |
| 容错 | `max_retries` | 任务失败时的最大重试次数。 | `.options(max_retries=3)` |
| 生命周期 | `max_calls` | (仅 Task) Worker 执行该任务的最大次数，之后重启 Worker 以释放内存。 | `.options(max_calls=100)` |



## 🚀 典型应用场景与代码示例
### 1. 动态资源调整 (Dynamic Resource Allocation)
这是 options 最直接的用途。你可以根据输入数据的大小，动态决定分配多少资源。

In [ ]:
@ray.remote
def process_data(data):
    # 处理数据
    return sum(data)

# 小任务：默认资源
small_task = process_data.remote([1, 2, 3])

# 大任务：动态申请更多 CPU
large_data = list(range(1000000))
big_task = process_data.options(num_cpus=8).remote(large_data)

ray.get([small_task, big_task])

### 2. 多返回值 (Multiple Returns)
默认情况下，Ray 函数只返回一个 ObjectRef。使用 options(num_returns=N) 可以让函数返回 N 个独立的 ObjectRef。

In [ ]:
@ray.remote
def get_coords():
    return 1, 2, 3  # 返回一个元组

# 方式 A：默认行为，x 是一个包含 (1,2,3) 的 ObjectRef
x = get_coords.remote() 

# 方式 B：使用 options，返回 3 个独立的 ObjectRef
x, y, z = get_coords.options(num_returns=3).remote()
# 此时 x, y, z 都是独立的 ObjectRef，可以单独 ray.get

### 3. 配合 Placement Group (PG) 进行精准调度
这是你在 OpenRLHF 代码中看到的高级用法。通过 options 将任务绑定到特定的资源束（Bundle）。

In [ ]:
# 假设 pg 是已经创建好的 Placement Group
# 强制将该 Actor 调度到 pg 的第 0 号 Bundle 上
actor = MyActor.options(
    scheduling_strategy=ray.util.scheduling_strategies.PlacementGroupSchedulingStrategy(
        placement_group=pg,
        placement_group_bundle_index=0
    )
).remote()

### 4. 动态生成器 (Dynamic Generators)
当返回值的数量在调用前无法确定时（例如流式处理），可以使用 num_returns="dynamic"。

In [ ]:
@ray.remote(num_returns="dynamic")
def generate_chunks(data):
    for chunk in data:
        yield chunk

# 返回的是一个生成器对象，可以迭代获取 ObjectRef
ref_gen = generate_chunks.remote(my_data)

## 💡 总结
@ray.remote options 是 Ray 灵活性的体现。

- 装饰器参数适合定义“默认配置”。
- .options() 方法适合定义“运行时特例”。

这种设计模式（类似于 Python 的 functools.partial 但更强大）使得 Ray 能够在保持代码简洁的同时，应对极其复杂的分布式调度需求。